In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from deap_alpha import Context
from deap_alpha import *
from datapreprocess import CSVDataLoader,DataFrameLoader,MultiAssetDataHandler
import warnings
import os
import re
import joblib
from deap_alpha.ops import *
from worldquant_utils.wqbrain_client import BrainBatchAlpha
from copy import deepcopy
from deap import algorithms
warnings.simplefilter("ignore")

MAX_TREE_DEPTH=6
MIN_TREE_DEPTH=2

context = Context()
# df = pd.read_feather(r"F:\MyCryptoTrading\CryptoTradingSystem_allin1\crypto_1h_klines_binancecryptoderivative_20251123_010959.feather")


def generate_mock_crypto_dataframe(
    symbols=("BTCUSDT", "ETHUSDT", "SOLUSDT"),
    start="2024-01-01 00:00:00",
    periods=10000,
    freq="1H",
    seed=42,
):
    """Create a reproducible synthetic OHLCV panel for multiple assets."""
    rng = np.random.default_rng(seed)
    timeline = pd.date_range(start=start, periods=periods, freq=freq)
    freq_delta = pd.to_timedelta(freq)
    rows = []

    for symbol in symbols:
        base_price = rng.uniform(50, 50000)
        price_walk = base_price + rng.normal(scale=base_price * 0.002, size=periods).cumsum()
        close_noise = rng.normal(scale=base_price * 0.001, size=periods)
        opens = price_walk
        closes = price_walk + close_noise
        highs = np.maximum(opens, closes) + rng.random(periods) * base_price * 0.0015
        lows = np.minimum(opens, closes) - rng.random(periods) * base_price * 0.0015
        volumes = rng.lognormal(mean=9, sigma=0.6, size=periods)
        taker_buy_base = volumes * rng.uniform(0.35, 0.75, size=periods)

        for idx, ts in enumerate(timeline):
            avg_price = 0.5 * (opens[idx] + closes[idx])
            quote_volume = volumes[idx] * avg_price
            taker_buy_quote = taker_buy_base[idx] * avg_price
            rows.append(
                {
                    "symbol": symbol,
                    "open_time": ts,
                    "open": float(opens[idx]),
                    "high": float(highs[idx]),
                    "low": float(lows[idx]),
                    "close": float(closes[idx]),
                    "volume": float(volumes[idx]),
                    "close_time": ts + freq_delta,
                    "quote_volume": float(quote_volume),
                    "trades": int(rng.integers(90, 1200)),
                    "taker_buy_base": float(taker_buy_base[idx]),
                    "taker_buy_quote": float(taker_buy_quote),
                }
            )

    df = pd.DataFrame(rows)
    df.sort_values(["symbol", "open_time"], inplace=True, ignore_index=True)
    return df


df = generate_mock_crypto_dataframe()
print(df.head())

handler = MultiAssetDataHandler(context=context,multi=True)
handler.add_loader(
    DataFrameLoader,
    [df]
)

data_3d, fields, stocks, dates, returns_matrix,context =\
      handler.to_3d_array(period=1,fields=
        ['open', 'high', 'low', 'close', 'volume',
       'quote_volume', 'trades', 'taker_buy_base',
       'taker_buy_quote']
      ) 


def compile_factor_matrix(individual):
    global data_3d
    func = toolbox.compile(expr=individual)
    # 计算因子值
    factor_matrix = (func(*data_3d)).transpose((1, 0))
    return factor_matrix


def evaluate_function(individual):
    global returns_matrix
    factor_matrix = compile_factor_matrix(individual)
    # 计算IC均值
    ic_mean = pd.DataFrame(factor_matrix).corrwith(
        pd.DataFrame(returns_matrix), method='spearman',axis=1).mean()
    # 计算因子值
    if ic_mean is None or np.isnan(ic_mean):
        ic_mean = -np.inf
    
    return ic_mean,


settings_dict = easy_initialize_gpsettings(
    context=context,
    min_depths=MIN_TREE_DEPTH,
    max_depths=MAX_TREE_DEPTH,
    fitness_weights=(1.0,),
    tournsize=5,
    mutate_max=MAX_TREE_DEPTH - MIN_TREE_DEPTH,
    mate_max=MAX_TREE_DEPTH - MIN_TREE_DEPTH,
    expr_mut_range=(0, MAX_TREE_DEPTH - MIN_TREE_DEPTH),
    use_cross_section=True,
    use_timeseries=True,
    wq_operators=False,
)

toolbox = settings_dict["toolbox"]
hof = settings_dict["hof"]
mstats = settings_dict["mstats"]
pset = settings_dict["pset"]
creator = settings_dict["creator"]

pop = toolbox.population(n=20)

    symbol           open_time          open          high           low  \
0  BTCUSDT 2024-01-01 00:00:00  38628.590918  38718.114747  38571.061341   
1  BTCUSDT 2024-01-01 01:00:00  38686.689506  38701.507479  38601.476562   
2  BTCUSDT 2024-01-01 02:00:00  38759.506342  38807.987242  38721.745118   
3  BTCUSDT 2024-01-01 03:00:00  38608.460691  38692.631191  38589.281593   
4  BTCUSDT 2024-01-01 04:00:00  38507.648286  38552.915043  38462.411320   

          close        volume          close_time  quote_volume  trades  \
0  38663.395183   6689.642995 2024-01-01 01:00:00  2.585279e+08     972   
1  38629.699229  10903.076221 2024-01-01 02:00:00  4.214932e+08    1016   
2  38748.462706   8297.730418 2024-01-01 03:00:00  3.215701e+08     823   
3  38640.556473  10171.896791 2024-01-01 04:00:00  3.928845e+08     191   
4  38510.187609   5696.176052 2024-01-01 05:00:00  2.193536e+08     868   

   taker_buy_base  taker_buy_quote  
0     4563.356395     1.763554e+08  
1     6155.292032 

In [3]:
ind_1 = "ts_rank(open,6)"
ind_1 = gp.PrimitiveTree.from_string(ind_1, pset)
ind_1 = creator.Individual(ind_1)

In [7]:
factor_matrix = compile_factor_matrix(ind_1)

In [13]:
factor_matrix.shape

(9998, 3)

In [ ]:
factor_matrix

array([[nan, nan, nan],
       [nan, nan, nan],
       [nan, nan, nan],
       ...,
       [ 5.,  5.,  6.],
       [ 6.,  2.,  6.],
       [ 6.,  1.,  6.]])

In [12]:
returns_matrix.shape

(9998, 3)

In [11]:
pd.DataFrame(factor_matrix).corrwith(
        pd.DataFrame(returns_matrix), method='spearman',axis=1)

0            NaN
1            NaN
2            NaN
3            NaN
4            NaN
          ...   
9993   -0.866025
9994    0.000000
9995    0.000000
9996    0.000000
9997    0.000000
Length: 9998, dtype: float64

In [6]:
evaluate_function(ind_1)

(-0.007782270828314865,)

In [3]:
data = pd.read_feather(r"F:\MyCryptoTrading\CryptoTradingSystem_allin1\crypto_1h_klines_binancecryptoderivative_20251123_010959.feather")

In [5]:
data.head(5).columns

Index(['symbol', 'open_time', 'open', 'high', 'low', 'close', 'volume',
       'close_time', 'quote_volume', 'trades', 'taker_buy_base',
       'taker_buy_quote'],
      dtype='object')